# Neural TSP - n=20 (RL-only)

Trains the Pointer Network with **reinforcement learning only** on n=20.

**Supervised pretraining is intentionally skipped.** Held-Karp computes exact optimal tours, but at n=20 it costs ~0.5s/instance, so 100k instances takes about **14h** (Colab kills it at ~12h). RL needs no labels - the model learns directly from the tour-length reward - so we go straight to policy-gradient training.

Estimated total on a T4: ~10-20 min (train) + ~15-30 min (eval).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 - Copy project & install deps

Also ensures `data/` exists: empty directories get dropped during Drive upload, which is the bug that broke data generation before.

In [ ]:
# Copy project to Colab local storage (faster than reading from Drive)
!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp

!pip install torch numpy matplotlib -q

# FIX: empty data/ is dropped on upload - recreate it so the C++ cells can write into it
import os
os.makedirs('/content/neural-tsp/data', exist_ok=True)
print('Project copied, data/ ready.')

## 2 - Generate n=20 data

Only the RL training set and eval set are needed - no `train_raw`/`test_raw`, since supervised labels are skipped.

In [ ]:
%%bash
cd /content/neural-tsp/cpp
set -e

g++ -std=c++17 -O2 generate_data.cpp -o generate_data

mkdir -p ../data   # belt-and-suspenders: ensure ../data exists before writing

./generate_data 100000 20 ../data/train_rl.txt
./generate_data 5000 20 ../data/eval.txt

ls -la ../data

## 3 - Save generated data to Drive

Backup so you don't regenerate next time.

In [ ]:
import shutil

shutil.copytree('/content/neural-tsp/data', '/content/drive/MyDrive/neural-tsp/data', dirs_exist_ok=True)
print('Data saved to Google Drive.')

## 4 - RL Actor-Critic training (n=20, from scratch)

Policy-gradient training with a learned critic baseline. Checkpoints are saved every 5 epochs and `train_rl.py` auto-resumes from the latest.

Any stale `model.pt` (e.g. an n=12 supervised stub) is removed first so the actor trains genuinely from scratch.

~5-10 min on T4.

In [ ]:
%cd /content/neural-tsp/python

# RL-only n=20: train from scratch. Drop any stale supervised model.pt
# so the actor isn't warm-started on the wrong graph size.
import os
if os.path.exists('model.pt'):
    os.remove('model.pt')
    print('Removed stale model.pt - training actor from scratch.')

%run train_rl.py

## 5 - Save RL models & checkpoints to Drive

Backup in case Colab disconnects.

In [ ]:
import shutil, os

for f in ['actor.pt', 'critic.pt']:
    src = f'/content/neural-tsp/python/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/neural-tsp/python/{f}')

ckpt_dir = '/content/neural-tsp/python/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.copytree(ckpt_dir, '/content/drive/MyDrive/neural-tsp/python/checkpoints', dirs_exist_ok=True)

print('Models and checkpoints saved to Google Drive.')

## 6 - Evaluate neural methods

Greedy / Sampling / Active Search on the eval set. The Sampling grid (especially N=12800 over 5000 instances) is the slow part - interrupt after Greedy + Active Search if you just want a quick check.

In [ ]:
%cd /content/neural-tsp/python
%run eval_rl.py

## 7 - Evaluate classical baselines

Nearest-Neighbor and 2-Opt, for comparison.

In [ ]:
%%bash
cd /content/neural-tsp/cpp
set -e
g++ -std=c++17 -O2 baselines.cpp -o baselines
./baselines < ../data/eval.txt

## 8 - Final backup to Drive

In [ ]:
import shutil, os

for f in ['actor.pt', 'critic.pt']:
    src = f'/content/neural-tsp/python/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/neural-tsp/python/{f}')

ckpt_dir = '/content/neural-tsp/python/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.copytree(ckpt_dir, '/content/drive/MyDrive/neural-tsp/python/checkpoints', dirs_exist_ok=True)

shutil.copytree('/content/neural-tsp/data', '/content/drive/MyDrive/neural-tsp/data', dirs_exist_ok=True)
print('Everything saved to Google Drive.')

---

## Resume after disconnection

If Colab disconnects during RL training, run these to resume from the last checkpoint.

### Resume 1 - Re-mount & copy project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp/python

### Resume 2 - Check available checkpoints

In [ ]:
import os

if os.path.exists('checkpoints'):
    ckpts = sorted([f for f in os.listdir('checkpoints') if f.endswith('.pt')])
    print(f'Available checkpoints: {ckpts}')
    print(f'Latest: {ckpts[-1]}')
else:
    print('No checkpoints found.')

### Resume 3 - Continue RL training

`train_rl.py` auto-detects the latest checkpoint and resumes from it.

In [ ]:
%run train_rl.py

---

## Switching to n=50

For n=50, regenerate data at n=50 in step 2 (`./generate_data 100000 50 ../data/train_rl.txt` and `... 50 ../data/eval.txt`) and re-run from step 4. Held-Karp stays skipped (infeasible). RL training takes ~45-80 min instead of ~5-10.